In [ ]:
from rmgpy import settings
from rmgpy.data.rmg import RMGDatabase
from base64 import b64encode
from IPython.display import display, HTML
import rmgpy
import numpy as np
from rmgpy.molecule.molecule import *
from rmgpy.molecule.group import Group
from rmgpy.species import *
from rmgpy.chemkin import *
from rmgpy.data.rmg import RMGDatabase
from IPython.display import display
from rmgpy.data.thermo import ThermoLibrary
from rmgpy.rmg.react import react
from rmgpy.species import Species
from rmgpy.reaction import Reaction
from rmgpy.data.rmg import get_db
from rmgpy.molecule.group import Group
from rmgpy.kinetics.arrhenius import ArrheniusBM
from rmgpy import settings
import time
from arkane.main import get_git_commit
import matplotlib.pyplot as plt
import matplotlib
import os
import sys
from rmgpy.data.kinetics.family import information_gain

In [ ]:
thermolibs = [
    'NCSU_2025_revised', 
    'C1_C2_Fluorine', #putting Siddha's as most trusted because Caroline used this thermo for calcs
    'PFCA_thermo',
    'Fluorine',
    'primaryThermoLibrary',
    'FFCM1(-)',
    'halogens',
    'CHOF_G4',
    'CHOCl_G4',
    'CHOBr_G4',
    'CHOFCl_G4',
    'CHOFBr_G4',
    'CHOFClBr_G4',
    'DFT_QCI_thermo',
    '2-BTP_G4',
    'thermo_DFT_CCSDTF12_BAC',
    'SulfurHaynes'
    ]

families = ['R_Recombination']

database = RMGDatabase()
database.load(
    path = settings['database.directory'],
    thermo_libraries = thermolibs,  # Can add others if necessary
    kinetics_families = families,
    reaction_libraries = [],
    kinetics_depositories = ['training'],
)

In [ ]:
family = database.kinetics.families[families[0]]

family.clean_tree()
root_node = family.groups.entries['Root']


parent_group_causing_split_violation = Group().from_adjacency_list("""1 * C   u1 r0 {3,[S,B,D,T,Q]}
2 * R   u1 {4,[S,B,D,T,Q]}
3   F   ux {1,[S,B,D,T,Q]}
4   R!H ux {2,[S,B,D,T,Q]}
""")

family.add_entry(root_node, parent_group_causing_split_violation, 'split_violation_node')


templateRxnMap = family.get_reaction_matches(thermo_database=database.thermo,remove_degeneracy=True, get_reverse=True,exact_matches_only=False,fix_labels=True)

#are the two reactions that we need in here? 
for tr_rxn in templateRxnMap['split_violation_node']:
    #display(tr_rxn)
    if 'CH2F + CH2F' in str(tr_rxn):
        CH2F_rxn = tr_rxn
    if 'CF + CF' in str(tr_rxn): 
        CF_rxn = tr_rxn
        
display(CH2F_rxn)
display(CF_rxn)

#now redefine the templateRxnMap so this split violation node only holds our two reactions of interest
templateRxnMap['split_violation_node'] = [CH2F_rxn, CF_rxn]

In [ ]:
split_violation_node_group = family.groups.entries['split_violation_node']

In [ ]:
exts, gave_up_split = family.get_extension_edge(split_violation_node_group, templateRxnMap, obj=information_gain, T = 1000, iter_max=10, iter_item_cap=100, leaf_node_max=2)

In [ ]:
exts

In [ ]:
gave_up_split

In [ ]:
family.extend_node(split_violation_node_group, templateRxnMap, iter_max=10, leaf_node_max=2)

#this exactly replicates the issue...now how to solve it??

#issue is that gave_up_split is not hit because len(grps)==iter+1 meaning we've iterated through all of the groups 

#there should be another flag that maybe says, if you've no extensions to further extend node with and you've exhausted all the extensions you could consider

In [ ]:
family = database.kinetics.families[families[0]]


family.clean_tree()

start = time.time()
family.generate_tree(thermo_database=database.thermo,
                     nprocs=1,
                     new_fraction_threshold_to_reopt_node=0.25,
                     max_batch_size=800,
                     extension_iter_max=2,
                     extension_iter_item_cap=100,
                    leaf_node_max=2)
end = time.time()
print(end-start)
print('generated tree')

In [ ]:
#why is it not suggesting an atomExt with 5 being an H? 
#why is it instead only suggesting a R!H connected to every atom? 
extension_with_H = Group().from_adjacency_list("""1 * C   u1 r0 {3,[S,B,D,T,Q]}
2 * R   u1 {4,[S,B,D,T,Q]} {5,[S,B,D,T,Q]}
3   F   ux {1,[S,B,D,T,Q]}
4   R!H ux {2,[S,B,D,T,Q]}
5   H   u0 {2,[S,B,D,T,Q]}
""")

name = 'trying_this'
val, boo = family.eval_ext(split_violation_node_group, extension_with_H, name, templateRxnMap, obj=information_gain, T = 1000)

In [ ]:
print(val, boo)
#this new extension does split the reactions
#why was this not suggested? 

In [ ]:
extension_with_H.get_extensions?

In [ ]:
family.generate_reactions(reactants=[r.molecule[0] for r in CH2F_rxn.reactants], products=[p.molecule[0] for p in CH2F_rxn.products])